# Lesson 8 — Embedding From Scratch

## 学习目标

语言模型接收的原始输入通常不是浮点向量，而是整数 Token ID。

例如一句话经过 tokenizer 后可能得到：

[15, 302, 91, 7]

这些数字只是 Vocabulary 中的编号。

神经网络不能直接把 Token ID 当作有语义的连续向量，因此需要：

Embedding

把：

$$
Token\ ID
$$

映射为：

$$
D
$$

维向量。

完成本节后，应能够：

1. 理解 Vocabulary 与 Token ID；
2. 理解 Embedding Matrix 的 Shape；
3. 理解 Embedding 本质上是 Row Lookup；
4. 推导单个 Token 的 Embedding Shape；
5. 推导 Sequence 的 Embedding Shape；
6. 推导 Batch Sequence 的 Embedding Shape；
7. 理解：

$$
(B,T)
\rightarrow
(B,T,D)
$$

是怎样发生的；

8. 理解 Embedding 与 Linear 的区别；
9. 使用 `nn.Module` 和 `nn.Parameter` 自己实现 Embedding；
10. 理解为什么 Token ID 必须是整数；
11. 使用 Autograd 检查 Embedding Gradient；
12. 与官方 `nn.Embedding` 做 Reference Test。

这一节最终实现：

`class Embedding(nn.Module)`

核心操作不是矩阵乘法，而是：

> 根据 Token ID，从一个可训练矩阵中取出对应的行。


## 1. Token ID

语言模型首先通过 tokenizer 把文本转换成整数编号。

例如：

文本：

`I love transformers`

可能被转换为：

$$
[31, 205, 917]
$$

这里的：

31,
205,
917

不是 token 的数值大小。

它们只是：

> Vocabulary 中的索引。

假设 Vocabulary Size：

$$
V=50000
$$

那么合法 Token ID 通常满足：

$$
0\le token\_id<V
$$

例如：

$$
token\_id=205
$$

表示：

> Vocabulary 中编号为 205 的 token。

因此不要把 Token ID 当成具有连续数值意义的 feature。

例如：

Token 100

并不意味着它比：

Token 50

“大两倍”。

它们只是不同类别的编号。


In [1]:
import torch

token_ids = torch.tensor([31, 205, 917], dtype=torch.long)

print("token_ids:", token_ids)
print("shape:", token_ids.shape)
print("dtype:", token_ids.dtype)

token_ids: tensor([ 31, 205, 917])
shape: torch.Size([3])
dtype: torch.int64


## 2. Vocabulary

Vocabulary 可以理解为 tokenizer 能够使用的全部 token 集合。

假设：

$$
V=10000
$$

表示整个 Vocabulary 中共有：

$$
10000
$$

个 Token。

它们对应 Token ID：

$$
0,1,2,\dots,9999
$$

如果模型的 hidden dimension 为：

$$
D=512
$$

那么我们希望每一个 Token ID 都拥有一个：

$$
512
$$

维向量。

因此需要存储：

$$
10000
$$

个向量。

每个向量：

$$
512
$$

维。

自然就得到一个矩阵：

$$
E\in\mathbb{R}^{V\times D}
$$

这个矩阵就是：

Embedding Matrix。


## 3. Embedding Matrix

Embedding Matrix：

$$
E.shape=(V,D)
$$

其中：

- $V$：Vocabulary Size；
- $D$：Embedding / Model Dimension。

可以把它想象成：

$$
E=
\begin{bmatrix}
---e_0---\\
---e_1---\\
---e_2---\\
\vdots\\
---e_{V-1}---
\end{bmatrix}
$$

每一行：

$$
e_i\in\mathbb{R}^{D}
$$

对应一个 Token ID。

因此：

Token ID：

$$
i
$$

对应：

$$
E[i]
$$

也就是 Embedding Matrix 的第 $i$ 行。

所以 Embedding 最核心的操作就是：

$$
token\_id
\rightarrow
E[token\_id]
$$


In [2]:
V = 10
D = 4

embedding_matrix = torch.randn(V, D)

print("Embedding Matrix Shape:", embedding_matrix.shape)

Embedding Matrix Shape: torch.Size([10, 4])


## 4. 单个 Token 的 Embedding

假设：

$$
E.shape=(V,D)
$$

Token ID：

$$
token\_id=3
$$

Embedding：

$$
E[3]
$$

根据 Lesson 1 的 Indexing Rule：

整数索引会删除对应维度。

原来：

$$
(V,D)
$$

第 0 维使用整数索引：

$$
3
$$

所以：

$$
E[3].shape=(D)
$$

因此：

$$
scalar\ token\ id
\rightarrow
(D)
$$

例如：

$$
D=4
$$

那么一个 token 最终表示为：

$$
(4)
$$

维向量。


In [3]:
V = 10
D = 4

embedding_matrix = torch.randn(V, D)

token_id = 3
embedding = embedding_matrix[token_id]

print("matrix:", embedding_matrix.shape)
print("token embedding:", embedding.shape)

matrix: torch.Size([10, 4])
token embedding: torch.Size([4])


## 5. 一整个 Token Sequence

假设一个 sequence：

$$
token\_ids=
[2,5,1]
$$

Shape：

$$
(T)
$$

其中：

$$
T=3
$$

Embedding Matrix：

$$
E.shape=(V,D)
$$

执行：

$$
E[token\_ids]
$$

相当于一次取出：

- 第 2 行；
- 第 5 行；
- 第 1 行。

每一行 Shape：

$$
(D)
$$

总共有：

$$
T
$$

行。

因此：

$$
(T)
\rightarrow
(T,D)
$$

这就是一个 token sequence 被转换成一组 embedding vectors。


In [4]:
V = 10
D = 4

embedding_matrix = torch.randn(V, D)

token_ids = torch.tensor([2, 5, 1], dtype=torch.long)
embeddings = embedding_matrix[token_ids]

print("token_ids:", token_ids.shape)
print("embeddings:", embeddings.shape)

token_ids: torch.Size([3])
embeddings: torch.Size([3, 4])


## 6. Shape Thinking

输入：

$$
token\_ids.shape=(T)
$$

其中每一个位置包含一个 Token ID。

对于每一个：

$$
token\_ids[t]
$$

Embedding lookup 会返回：

$$
(D)
$$

所以：

$$
T
$$

个 token：

每一个都增加一个：

$$
D
$$

维 embedding。

因此：

$$
(T)
\rightarrow
(T,D)
$$

可以形成一个简单规则：

> Embedding 会在 Token ID Tensor 的末尾增加一个 Embedding Dimension。

例如：

$$
(3)
\rightarrow
(3,4)
$$

其中：

3

仍然表示 Sequence Length。

4

是新增的 Embedding Dimension。


## 7. Batch 中的 Token IDs

语言模型训练时输入通常是：

$$
token\_ids.shape=(B,T)
$$

例如：

$$
(2,3)
$$

可以理解为：

2 个 sequence，

每个 sequence 有 3 个 token。

Embedding Matrix：

$$
E.shape=(V,D)
$$

对：

$$
(B,T)
$$

中的每一个 Token ID 做 row lookup。

每个 Token ID 得到：

$$
(D)
$$

因此：

$$
(B,T)
\rightarrow
(B,T,D)
$$

这是语言模型中最重要的 Shape 变化之一。


In [5]:
B = 2
T = 3
V = 10
D = 4

embedding_matrix = torch.randn(V, D)

token_ids = torch.tensor([[2, 5, 1], [7, 3, 4]], dtype=torch.long)
embeddings = embedding_matrix[token_ids]

print("token_ids:", token_ids.shape)
print("embedding_matrix:", embedding_matrix.shape)
print("embeddings:", embeddings.shape)

token_ids: torch.Size([2, 3])
embedding_matrix: torch.Size([10, 4])
embeddings: torch.Size([2, 3, 4])


## 8. Batch Embedding 本质上仍然是 Row Lookup

对于：

$$
token\_ids.shape=(B,T)
$$

最终：

$$
output[b,t]
$$

就是：

$$
E[token\_ids[b,t]]
$$

例如：

$$
token\_ids[0,1]=5
$$

那么：

$$
output[0,1]=E[5]
$$

所以 Embedding 并没有执行复杂的矩阵运算。

它只是对输入中的每一个整数 ID：

> 找到 Embedding Matrix 对应的一行。


In [6]:
B = 2
T = 3
V = 10
D = 4

embedding_matrix = torch.randn(V, D)

token_ids = torch.tensor([[2, 5, 1], [7, 3, 4]], dtype=torch.long)

output = embedding_matrix[token_ids]
token = token_ids[0, 1]

manual = embedding_matrix[token]
automatic = output[0, 1]

print("token id:", token.item())
print("same:", torch.allclose(manual, automatic))

token id: 5
same: True


## 9. Embedding Shape Rule

如果 Token ID Tensor：

$$
input.shape=(d_1,d_2,\dots,d_n)
$$

Embedding Dimension：

$$
D
$$

那么 Embedding 输出：

$$
output.shape=
(d_1,d_2,\dots,d_n,D)
$$

也就是说：

> 保留输入所有维度，并在最后增加 Embedding Dimension。

例如：

单个 Token：

$$
()
\rightarrow
(D)
$$

Sequence：

$$
(T)
\rightarrow
(T,D)
$$

Batch Sequence：

$$
(B,T)
\rightarrow
(B,T,D)
$$

更一般地：

$$
(...)
\rightarrow
(...,D)
$$

这个 Shape Rule 以后看到 `nn.Embedding` 时应该形成条件反射。


## 10. Embedding vs Linear

这是这一节非常重要的区别。

Linear：

输入：

$$
(...,D_{in})
$$

通过矩阵乘法：

$$
XW^T+b
$$

得到：

$$
(...,D_{out})
$$

它处理的是：

> Floating-point Feature Vector。

---

Embedding：

输入：

$$
(...)
$$

其中元素是：

> Integer Token IDs。

它不计算：

$$
XW
$$

而是直接：

$$
E[token\_id]
$$

所以 Embedding 本质是：

> Table Lookup / Row Lookup。

例如：

Token ID：

$$
5
$$

直接选择：

$$
E[5]
$$

而不是把数字 5 当成连续数值送入 Linear Layer。


## 11. Token ID 没有连续数值语义

假设：

Token A：

$$
id=10
$$

Token B：

$$
id=20
$$

Token C：

$$
id=30
$$

这些数字只是 Vocabulary Index。

不能推出：

$$
TokenC
=
TokenA+TokenB
$$

也不能说：

$$
Token\ 20
$$

在语义上是：

$$
Token\ 10
$$

的两倍。

因此 Token ID 不适合作为普通连续 feature 直接参与：

$$
WX+b
$$

Embedding 的作用就是：

> 为每一个离散 Token ID 学习一个独立的连续向量表示。

从：

Discrete ID

↓

Learnable Vector

完成从离散符号到连续表示空间的转换。


## 12. Embedding Matrix 需要学习

Embedding Matrix：

$$
E.shape=(V,D)
$$

不是固定查找表。

它是模型参数。

训练开始时：

$$
E
$$

通常被随机初始化。

训练过程中：

`loss.backward()`

会计算：

$$
\frac{\partial L}{\partial E}
$$

然后 Optimizer 更新：

$$
E
$$

所以每个 Token 的 Embedding Vector 会随着训练不断变化。

因此：

Embedding Matrix

不是 tokenizer 自带的语义表。

而是：

> 模型通过训练学习出来的 Parameter。


## 13. Embedding Parameter Count

Embedding Matrix：

$$
E.shape=(V,D)
$$

没有额外 Bias。

因此参数总数：

$$
N=V\times D
$$

例如：

$$
V=50000
$$

$$
D=768
$$

那么参数量：

$$
50000\times768
$$

等于：

$$
38,400,000
$$

也就是约：

$$
38.4M
$$

Parameters。

因此对于大型 Vocabulary：

> Token Embedding 本身就可能占据大量模型参数。


In [7]:
V = 50_000
D = 768

num_parameters = V * D

print("Embedding parameters:", num_parameters)

Embedding parameters: 38400000


## 14. Embedding From Scratch

现在自己实现：

`class Embedding(nn.Module)`

Module 需要保存一个 Parameter：

$$
weight.shape=(V,D)
$$

其中：

- 每一行对应一个 Token；
- 每一列对应一个 Embedding Feature。

因此：

`weight[token_id]`

就能得到对应 Token Vector。

Forward：

输入：

$$
token\_ids.shape=(...)
$$

执行：

`self.weight[token_ids]`

输出：

$$
(...,D)
$$

所以我们的 Embedding 结构非常简单：

Embedding

↓

Weight Parameter

$$
(V,D)
$$

↓

Integer Indexing

↓

Output

$$
(...,D)
$$


In [8]:
from torch import nn


class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int) -> None:
        super().__init__()

        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim

        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids]

## 15. Embedding Weight Shape

创建：

`Embedding(10000, 512)`

表示：

Vocabulary Size：

$$
V=10000
$$

Embedding Dimension：

$$
D=512
$$

所以 Weight：

$$
weight.shape=(10000,512)
$$

Parameter Count：

$$
10000\times512
$$

由于：

`weight`

使用：

`nn.Parameter`

创建，

它也应该自动出现在：

`named_parameters()`

中。


In [9]:
embedding = Embedding(10_000, 512)

print("num_embeddings:", embedding.num_embeddings)
print("embedding_dim:", embedding.embedding_dim)
print("weight.shape:", embedding.weight.shape)

num_embeddings: 10000
embedding_dim: 512
weight.shape: torch.Size([10000, 512])


In [10]:
embedding = Embedding(100, 32)

for name, parameter in embedding.named_parameters():
    print(name, parameter.shape)


weight torch.Size([100, 32])


## 17. Forward Shape Test

假设：

$$
B=2
$$

$$
T=4
$$

Vocabulary：

$$
V=100
$$

Embedding Dimension：

$$
D=16
$$

输入：

$$
token\_ids.shape=(2,4)
$$

Weight：

$$
weight.shape=(100,16)
$$

Forward：

`weight[token_ids]`

对于输入中的每一个 Token ID，都取出一个：

$$
(16)
$$

维向量。

因此：

$$
(2,4)
\rightarrow
(2,4,16)
$$


In [11]:
B = 2
T = 4
V = 100
D = 16

embedding = Embedding(V, D)

token_ids = torch.tensor([[1, 5, 9, 3], [7, 2, 8, 4]], dtype=torch.long)
output = embedding(token_ids)

print("token_ids:", token_ids.shape)
print("weight:", embedding.weight.shape)
print("output:", output.shape)

token_ids: torch.Size([2, 4])
weight: torch.Size([100, 16])
output: torch.Size([2, 4, 16])


## 18. `torch.empty()` 仍然需要初始化

和上一节 Linear 一样：

`torch.empty(...)`

只是分配内存。

当前：

`self.weight`

虽然：

- Shape 正确；
- Parameter Registration 正确；
- Forward Indexing 正确；

但 Weight 还没有经过有效初始化。

因此完整 Embedding Module 还需要：

`reset_parameters()`

Embedding 常见做法之一是使用随机正态分布初始化。

例如：

$$
E_{ij}
\sim
\mathcal{N}(0,1)
$$

当前阶段重点不是初始化分布本身。

重点仍然是：

> Trainable Parameter 创建之后必须拥有明确的初始化策略。

下一步我们会把初始化加入 Module，然后与官方：

`nn.Embedding`

进行 Forward 和 Backward Reference Test。


In [ ]:
embedding = Embedding(10, 4)

print(embedding.weight)

Parameter containing:
tensor([[-4.5404e-35,  3.4514e-41,  0.0000e+00,  0.0000e+00],
        [-9.5334e-01, -4.9461e-01,  2.1620e-01,  1.6603e+00],
        [-4.0612e-01,  1.3428e+00,  7.8250e-01, -1.4633e+00],
        [ 6.1572e-02,  6.2190e-01,  4.1039e-02, -1.2227e+00],
        [ 2.8831e-01,  1.0438e+00,  4.0671e-01,  7.8296e-01],
        [-9.6034e-01,  7.7314e-01,  7.9057e-01,  1.7011e+00],
        [-3.1360e-01,  6.2437e-01,  9.0469e-01,  2.4435e-01],
        [-8.5374e-01, -1.7685e+00, -1.1184e+00, -6.7337e-02],
        [-5.5638e-01, -5.0220e-01, -1.3961e-01,  9.2795e-01],
        [ 1.4567e+00, -3.1269e+00, -1.0651e+00, -2.2509e-01]],
       requires_grad=True)


## 19. Embedding Parameter Initialization

上一部分我们已经实现了 Embedding 最核心的 Forward：

`weight[token_ids]`

其中：

$$
weight.shape=(V,D)
$$

输入：

$$
token\_ids.shape=(B,T)
$$

输出：

$$
(B,T,D)
$$

但是目前还有一个问题：

`torch.empty(...)`

只负责分配内存，并不会提供适合训练的参数初始化。

因此一个完整的 Trainable Module 应该显式初始化 Parameter。

对于 Embedding，可以使用正态分布初始化：

$$
E_{ij}\sim\mathcal{N}(0,1)
$$

其中：

- $V$：Vocabulary Size
- $D$：Embedding Dimension
- $E\in\mathbb{R}^{V\times D}$：Embedding Matrix

目前阶段不需要纠结“哪一种初始化一定最好”。

真正需要建立的工程习惯是：

> 创建 Trainable Parameter 后，应明确知道它如何被初始化。


In [15]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int) -> None:
        super().__init__()

        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim

        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.weight, mean=0.0, std=1.0)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids]


## 20. 检查初始化结果

现在：

`Embedding(V, D)`

创建后：

$$
weight.shape=(V,D)
$$

并且每个参数已经经过显式随机初始化。

例如：

$$
V=100,\quad D=16
$$

则：

$$
weight.shape=(100,16)
$$

Parameter 数量：

$$
100\times16=1600
$$

因此 Embedding 的参数量公式非常简单：

$$
N_{\text{embedding}}=V\times D
$$

这个公式以后在语言模型中非常重要。

例如：

$$
V=50\,000
$$

$$
D=4096
$$

仅 Token Embedding 就有：

$$
50\,000\times4096
=
204\,800\,000
$$

也就是大约：

$$
205M
$$

个参数。

因此 Vocabulary Size 不只是 Tokenizer 问题。

它也会直接影响：

- Embedding 参数量；
- LM Head 参数量；
- 显存；
- 模型计算量。


In [16]:
embedding = Embedding(num_embeddings=100, embedding_dim=16)

print("weight shape:", embedding.weight.shape)
print("parameter count:", embedding.weight.numel())
print("mean:", embedding.weight.mean().item())
print("std:", embedding.weight.std().item())

weight shape: torch.Size([100, 16])
parameter count: 1600
mean: 0.003726525232195854
std: 0.9980543851852417


## 21. Token ID 的 Dtype

Embedding 的输入不是普通连续特征。

它是：

> Vocabulary 中的离散索引。

例如：

$$
token\_ids=
[1,5,9,3]
$$

这些数字的语义不是：

- 数值大小；
- 距离；
- 连续 feature。

而是：

> 第 1、5、9、3 行 Embedding Vector。

因此 Embedding lookup 本质是索引操作。

PyTorch 中最常用的 Token ID dtype 是：

`torch.long`

即：

`torch.int64`

例如：

`token_ids.dtype == torch.int64`

这和语言模型中常见的：

$$
X\in\mathbb{R}^{B\times T\times D}
$$

不同。

Token ID Tensor：

$$
token\_ids\in\mathbb{Z}^{B\times T}
$$

Embedding 输出才变成：

$$
X\in\mathbb{R}^{B\times T\times D}
$$

因此整个入口可以理解成：

$$
\text{Discrete IDs}
\rightarrow
\text{Continuous Vectors}
$$


In [17]:
token_ids = torch.tensor([[1, 5, 9, 3], [7, 2, 8, 4]], dtype=torch.long)

embedding = Embedding(num_embeddings=100, embedding_dim=16)

output = embedding(token_ids)

print("token_ids dtype:", token_ids.dtype)
print("token_ids shape:", token_ids.shape)
print("output dtype:", output.dtype)
print("output shape:", output.shape)

token_ids dtype: torch.int64
token_ids shape: torch.Size([2, 4])
output dtype: torch.float32
output shape: torch.Size([2, 4, 16])


## 22. Token ID 为什么不是 Float？

假设：

`token_id = 5`

它表示：

> 取 Embedding Matrix 的第 5 行。

但是：

`token_id = 5.37`

不存在“第 5.37 行”。

因此 Token ID 的数学角色是：

$$
i\in\{0,1,\ldots,V-1\}
$$

而不是：

$$
i\in\mathbb{R}
$$

Embedding 是一个离散 lookup operation。

这也是为什么：

- 模型不能直接对 Token ID 求梯度；
- 梯度流向的是 Embedding Weight；
- Token ID 本身只是决定“访问哪几行参数”。


In [18]:
embedding = Embedding(10, 4)

float_ids = torch.tensor([1.0, 2.0, 3.0])

try:
    output = embedding(float_ids)
except Exception as error:
    print(type(error).__name__)
    print(error)


IndexError
tensors used as indices must be long, int, byte or bool tensors


## 23. Token ID 的合法范围

如果 Vocabulary Size：

$$
V
$$

那么合法 Token ID 为：

$$
0,1,2,\ldots,V-1
$$

即：

$$
0\le token\_id<V
$$

例如：

$$
V=10
$$

合法：

`0 ... 9`

不合法：

`10`

因为 Embedding Matrix：

$$
weight.shape=(10,D)
$$

只有：

$$
0,\ldots,9
$$

这 10 行。

因此 Embedding lookup 同时隐含一个重要约束：

> Tokenizer 输出的 Token ID 必须与模型 Vocabulary Size 完全一致。

这会在后面 CS336 Tokenizer → Transformer 对接时非常重要。


In [19]:
embedding = Embedding(num_embeddings=10, embedding_dim=4)

valid_ids = torch.tensor([0, 3, 9], dtype=torch.long)

print(embedding(valid_ids).shape)

invalid_ids = torch.tensor([10], dtype=torch.long)

try:
    embedding(invalid_ids)
except IndexError as error:
    print(error)


torch.Size([3, 4])
index 10 is out of bounds for dimension 0 with size 10


## 24. 相同 Token ID 会得到相同 Embedding

Embedding Matrix：

$$
E\in\mathbb{R}^{V\times D}
$$

对于 Token ID：

$$
i
$$

输出：

$$
E_i
$$

因此，如果同一个 Token ID 出现多次：

$$
[5,5,5]
$$

那么 lookup 出来的三个向量完全相同：

$$
[E_5,E_5,E_5]
$$

注意这里描述的是：

> Token Embedding Lookup 本身。

Transformer 后面加入：

- Position Information；
- Attention；
- Context；

以后，同一个 token 在不同位置的 hidden state 通常就不会继续相同。

因此需要区分：

### Token Embedding

只由 Token ID 决定。

### Contextual Representation

经过 Transformer 后，还受到：

- position；
- surrounding tokens；
- attention；

等因素影响。


In [20]:
embedding = Embedding(num_embeddings=10, embedding_dim=4)

token_ids = torch.tensor([5, 5, 5], dtype=torch.long)

output = embedding(token_ids)

print(output)
print("0 == 1:", torch.equal(output[0], output[1]))
print("1 == 2:", torch.equal(output[1], output[2]))

tensor([[-1.2278, -3.0940, -0.8619, -1.5202],
        [-1.2278, -3.0940, -0.8619, -1.5202],
        [-1.2278, -3.0940, -0.8619, -1.5202]], grad_fn=<IndexBackward0>)
0 == 1: True
1 == 2: True


## 25. Embedding 不是固定查表

第一次接触 Embedding 时，很容易把它理解成：

> 一个固定 dictionary。

但神经网络中的 Embedding Matrix 实际是：

`nn.Parameter`

所以：

$$
E\in\mathbb{R}^{V\times D}
$$

中的元素会通过 Gradient Descent 学习。

训练过程：

$$
token\_ids
$$

$$
\downarrow
$$

$$
E[token\_ids]
$$

$$
\downarrow
$$

$$
Transformer
$$

$$
\downarrow
$$

$$
Loss
$$

$$
\downarrow backward
$$

$$
\frac{\partial L}{\partial E}
$$

因此 Embedding 向量不是人为规定：

> cat 应该对应哪个向量。

而是在训练过程中由 Loss 自动学习。


## 26. Embedding Lookup 如何参与 Backpropagation？

假设：

$$
E\in\mathbb{R}^{V\times D}
$$

输入只有一个 Token ID：

$$
i
$$

Forward：

$$
x=E_i
$$

Loss：

$$
L=f(x)
$$

Backward 时：

$$
\frac{\partial L}{\partial E}
$$

只有当前被访问的行：

$$
E_i
$$

会直接收到来自这个 lookup 的梯度。

其它没有被使用的行：

$$
E_j,\quad j\neq i
$$

没有经过当前计算路径，因此对应梯度通常为 0。

因此 Embedding Gradient 往往具有一种：

> sparse-looking structure

也就是：

虽然默认 `.grad` Tensor 本身可能是 dense Tensor，

但是许多未访问行的数值为 0。


In [21]:
torch.manual_seed(0)

embedding = Embedding(num_embeddings=6, embedding_dim=3)

token_ids = torch.tensor([1, 4], dtype=torch.long)

output = embedding(token_ids)

loss = output.sum()
loss.backward()

print("weight.grad shape:", embedding.weight.grad.shape)
print(embedding.weight.grad)

weight.grad shape: torch.Size([6, 3])
tensor([[0., 0., 0.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.],
        [1., 1., 1.],
        [0., 0., 0.]])


## 27. 手动分析一个 Embedding Gradient

假设：

$$
token\_ids=[1,4]
$$

Forward：

$$
output=
\begin{bmatrix}
E_1\\
E_4
\end{bmatrix}
$$

如果：

$$
L=\sum output
$$

那么对于每个被访问的 embedding element：

$$
\frac{\partial L}
{\partial E_{ij}}
=
1
$$

因此：

$$
\frac{\partial L}{\partial E_1}
=
[1,1,\ldots,1]
$$

以及：

$$
\frac{\partial L}{\partial E_4}
=
[1,1,\ldots,1]
$$

而没有被访问的：

$$
E_0,E_2,E_3,E_5
$$

对应梯度：

$$
0
$$

这说明一个非常重要的事实：

> Indexing operation 虽然是离散选择，但被选择出来的 Weight 仍然可以通过 Autograd 接收梯度。


## 28. Repeated Token 的 Gradient Accumulation

现在考虑：

$$
token\_ids=[2,2,2]
$$

同一个 Embedding Row：

$$
E_2
$$

在 Forward 中被使用了 3 次。

如果：

$$
L=\sum output
$$

每一次 lookup 都会为：

$$
E_2
$$

贡献梯度：

$$
[1,1,\ldots,1]
$$

因为 Autograd 会累加来自所有计算路径的梯度：

$$
\frac{\partial L}{\partial E_2}
=
[3,3,\ldots,3]
$$

这和上一节 Autograd 中学到的 Gradient Accumulation 是同一个原则。

因此：

> 同一个 Parameter 如果在计算图中被使用多次，所有路径产生的梯度会累加。


In [22]:
torch.manual_seed(0)

embedding = Embedding(num_embeddings=5, embedding_dim=4)

token_ids = torch.tensor([2, 2, 2], dtype=torch.long)

output = embedding(token_ids)

loss = output.sum()
loss.backward()

print(embedding.weight.grad)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [3., 3., 3., 3.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


## 29. 为什么必须做 Reference Test？

我们自己实现 Module 后，不应该通过：

> 看起来差不多

判断它是否正确。

更可靠的方法是：

> 和 PyTorch 官方实现对齐。

这里比较：

### Our Implementation

`Embedding`

### Reference Implementation

`torch.nn.Embedding`

关键是：

> 两个 Module 必须使用完全相同的 Weight。

否则即使实现完全正确，两边随机初始化不同，输出也必然不同。

测试思路：

1. 创建自己的 Embedding；
2. 创建 `nn.Embedding`；
3. 把相同 Weight 拷贝给两者；
4. 输入相同 Token IDs；
5. 比较 Forward Output。

如果：

`torch.allclose(...) == True`

说明 Forward 行为一致。


In [23]:
torch.manual_seed(0)

V = 10
D = 4

our_embedding = Embedding(V, D)

reference_embedding = nn.Embedding(V, D)

with torch.no_grad():
    reference_embedding.weight.copy_(our_embedding.weight)

token_ids = torch.tensor([[1, 3, 5], [2, 7, 3]], dtype=torch.long)

our_output = our_embedding(token_ids)
reference_output = reference_embedding(token_ids)

print("our output shape:", our_output.shape)
print("reference shape:", reference_output.shape)
print("same:", torch.allclose(our_output, reference_output))

our output shape: torch.Size([2, 3, 4])
reference shape: torch.Size([2, 3, 4])
same: True
